# CPU Jacobian-lens prompt playground

This notebook is the reusable, cell-by-cell version of `scripts/02_fit_smoke_lens_cpu.py`. It loads the local Qwen 2.5 3B model on CPU and fits the smoke-test lens only when no saved lens is available.

**Normal workflow**

1. Run sections 1–5 once.
2. Change only `evaluation_prompt` in the final cell.
3. Rerun only that final cell for every new prompt.

The model stays in notebook memory and the fitted lens stays in `artifacts/lenses/`. After a kernel restart you must reload the model, but the saved lens is loaded instead of being fitted again.

## 1. Imports and configuration

Keep `FORCE_REFIT = False` for normal prompt testing. Set it to `True` only when you intentionally want to rebuild the lens from scratch; fitting is the expensive step.

In [1]:
import logging
import os
import sys
import time
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Find the repository root whether Jupyter starts here or in scripts/."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "jlens").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the jacobian-lens repository root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import jlens
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = PROJECT_ROOT / "models/Qwen2.5-3B-Instruct"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/lenses"
CHECKPOINT_PATH = ARTIFACT_DIR / "qwen3b_cpu_smoke_fit_checkpoint.pt"
LENS_PATH = ARTIFACT_DIR / "qwen3b_cpu_smoke_lens.pt"

FORCE_REFIT = False

FIT_PROMPTS = [
    (
        "Artificial intelligence systems process natural language by "
        "transforming input tokens through multiple neural network layers. "
        "Intermediate representations encode different types of information "
        "before the model predicts the next token."
    )
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Model path   : {MODEL_PATH}")
print(f"Lens path    : {LENS_PATH}")
print(f"Saved lens   : {LENS_PATH.exists()}")

/home/khanhtran/Documents/Data-Impact/security-agentic/jacobian-lens-guard/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root : /home/khanhtran/Documents/Data-Impact/security-agentic/jacobian-lens-guard
Model path   : /home/khanhtran/Documents/Data-Impact/security-agentic/jacobian-lens-guard/models/Qwen2.5-3B-Instruct
Lens path    : /home/khanhtran/Documents/Data-Impact/security-agentic/jacobian-lens-guard/artifacts/lenses/qwen3b_cpu_smoke_lens.pt
Saved lens   : True


## 2. Configure the CPU environment

In [2]:
cpu_count = os.cpu_count() or 1
torch.set_num_threads(min(8, cpu_count))

print(f"Torch version : {torch.__version__}")
print(f"CUDA          : {torch.cuda.is_available()}")
print(f"CPU threads   : {torch.get_num_threads()}")

Torch version : 2.13.0+cpu
CUDA          : False
CPU threads   : 8


## 3. Load and wrap the model once

This is slow and uses substantial RAM. If you accidentally rerun this cell with the same model already loaded, it reuses the existing objects.

In [3]:
loaded_path = globals().get("_loaded_model_path")
model_is_ready = all(
    name in globals() for name in ("tokenizer", "hf_model", "model")
)

if model_is_ready and loaded_path == str(MODEL_PATH.resolve()):
    print("Model is already loaded in this kernel; reusing it.")
else:
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
    )

    print("Loading Qwen 3B in FP32 on CPU...")
    load_started = time.perf_counter()
    hf_model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        dtype=torch.float32,
        local_files_only=True,
    )
    hf_model.to("cpu")
    hf_model.eval()

    print(f"Model loaded in {time.perf_counter() - load_started:.2f} seconds")
    print("Wrapping the model with jlens.from_hf()...")
    model = jlens.from_hf(hf_model, tokenizer, compile=False)
    _loaded_model_path = str(MODEL_PATH.resolve())

source_layer = model.n_layers - 2
target_layer = model.n_layers - 1

print(model)
print(f"Number of layers : {model.n_layers}")
print(f"Residual width   : {model.d_model}")
print(f"Source layer     : {source_layer}")
print(f"Target layer     : {target_layer}")

Loading tokenizer...
Loading Qwen 3B in FP32 on CPU...


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 647.75it/s]

Model loaded in 0.86 seconds
Wrapping the model with jlens.from_hf()...
HFLensModel(Qwen2ForCausalLM, n_layers=36, d_model=2048)
Number of layers : 36
Residual width   : 2048
Source layer     : 34
Target layer     : 35


## 4. Load the saved lens, or fit it once

With the repository's existing artifact, this cell only loads the lens. If the lens file does not exist, it runs the same smoke fit as the Python script, resumes from its checkpoint when possible, and saves the result.

In [4]:
if LENS_PATH.is_file() and not FORCE_REFIT:
    print(f"Loading existing lens: {LENS_PATH}")
    lens = jlens.JacobianLens.load(str(LENS_PATH))
else:
    if FORCE_REFIT:
        print("FORCE_REFIT=True: fitting the lens from scratch.")
    else:
        print("No saved lens found; fitting it now.")

    print("Smoke fitting configuration:")
    print(f"  Prompts      : {len(FIT_PROMPTS)}")
    print("  Max seq len  : 24")
    print("  Skip first   : 2")
    print("  Dim batch    : 32")
    print(f"  Backward passes per prompt: {(model.d_model + 31) // 32}")

    fit_started = time.perf_counter()
    lens = jlens.fit(
        model,
        prompts=FIT_PROMPTS,
        source_layers=[source_layer],
        target_layer=target_layer,
        dim_batch=32,
        max_seq_len=24,
        skip_first=2,
        checkpoint_path=str(CHECKPOINT_PATH),
        checkpoint_every=1,
        resume=not FORCE_REFIT,
    )
    print(f"Fitting completed in {time.perf_counter() - fit_started:.2f} seconds")
    lens.save(str(LENS_PATH))
    print(f"Lens saved to: {LENS_PATH}")

if lens.d_model != model.d_model:
    raise ValueError(
        f"Lens width {lens.d_model} does not match model width {model.d_model}."
    )
if source_layer not in lens.source_layers:
    raise ValueError(
        f"Layer {source_layer} is not in saved lens layers {lens.source_layers}."
    )

print(f"Ready: {lens}")
print(f"Lens size: {LENS_PATH.stat().st_size / 1024**2:.2f} MiB")

Loading existing lens: /home/khanhtran/Documents/Data-Impact/security-agentic/jacobian-lens-guard/artifacts/lenses/qwen3b_cpu_smoke_lens.pt
Ready: JacobianLens(d_model=2048, n_prompts=1, source_layers=[34..34] (1 layers))
Lens size: 8.00 MiB


## 5. Define the reusable evaluation helper

Run this once. By default each evaluation compares the Jacobian lens, vanilla logit lens, and final model logits. Set `compare_logit_lens=False` in the final cell if you only need J-lens and model results; that avoids the second forward pass.

In [5]:
def decode_top_tokens(
    logits: torch.Tensor,
    top_k: int = 10,
) -> list[tuple[str, float]]:
    """Decode the highest-scoring tokens from one logits vector."""
    if top_k < 1:
        raise ValueError("top_k must be at least 1.")
    values, indices = logits.topk(min(top_k, logits.shape[-1]))
    return [
        (repr(tokenizer.decode([token_id])), score)
        for token_id, score in zip(
            indices.tolist(),
            values.tolist(),
            strict=True,
        )
    ]


def print_top_tokens(title: str, logits: torch.Tensor, top_k: int) -> None:
    print(f"\n{title}")
    for token, score in decode_top_tokens(logits, top_k=top_k):
        print(f"{token:>20}  {score:>10.4f}")


def evaluate_prompt(
    prompt: str,
    *,
    top_k: int = 10,
    position: int = -1,
    max_seq_len: int = 512,
    compare_logit_lens: bool = True,
) -> dict[str, object]:
    """Evaluate one prompt without reloading the model or refitting the lens."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a non-empty string.")

    started = time.perf_counter()
    jlens_logits, model_logits, input_ids = lens.apply(
        model,
        prompt,
        layers=[source_layer],
        positions=[position],
        max_seq_len=max_seq_len,
        use_jacobian=True,
    )

    logit_lens_logits = None
    if compare_logit_lens:
        logit_lens_logits, _, _ = lens.apply(
            model,
            prompt,
            layers=[source_layer],
            positions=[position],
            max_seq_len=max_seq_len,
            use_jacobian=False,
        )

    print("=== Evaluation prompt ===")
    print(prompt)
    print("\nInput tokens:")
    print(tokenizer.convert_ids_to_tokens(input_ids[0].tolist()))

    print_top_tokens(
        f"J-lens top tokens at layer {source_layer}:",
        jlens_logits[source_layer][0],
        top_k,
    )
    if logit_lens_logits is not None:
        print_top_tokens(
            f"Logit-lens top tokens at layer {source_layer}:",
            logit_lens_logits[source_layer][0],
            top_k,
        )
    print_top_tokens("Model final top tokens:", model_logits[0], top_k)
    print(f"\nEvaluation completed in {time.perf_counter() - started:.2f} seconds")

    return {
        "prompt": prompt,
        "input_ids": input_ids,
        "jlens_logits": jlens_logits,
        "logit_lens_logits": logit_lens_logits,
        "model_logits": model_logits,
    }


print("Evaluation helper is ready. Rerun only the final cell for new prompts.")

Evaluation helper is ready. Rerun only the final cell for new prompts.


## 6. Prompt evaluation — rerun only this cell

Edit `evaluation_prompt` and press **Shift+Enter**. Nothing is reloaded or refitted.

In [7]:
evaluation_prompt = "The capital city of Spain is"

evaluation_result = evaluate_prompt(
    evaluation_prompt,
    top_k=10,
    position=-1,
    compare_logit_lens=True,
)

=== Evaluation prompt ===
The capital city of Spain is

Input tokens:
['The', 'Ġcapital', 'Ġcity', 'Ġof', 'ĠSpain', 'Ġis']

J-lens top tokens at layer 34:
           ' Madrid'     18.8720
               ' __'     15.7011
           ' ______'     15.6180
              '____'     15.5043
             ' ____'     15.2753
          ' located'     15.1754
        ' Barcelona'     15.1704
             ':\n\n'     14.2694
               ':\n'     13.9555
              ' ___'     13.8648

Logit-lens top tokens at layer 34:
           ' Madrid'     17.1519
               ' __'     16.5356
             ' ____'     15.7532
           ' ______'     15.6724
              '____'     15.4999
          ' located'     14.4670
              ' ___'     14.2078
            ' _____'     13.7959
        ' Barcelona'     13.6578
          '________'     13.1162

Model final top tokens:
           ' Madrid'     20.2024
               ' __'     19.1773
           ' ______'     17.8784
             ' ____'     